In [0]:
import requests

# Widgets for the dummyjson.com/users API
# https://dummyjson.com/docs/users
dbutils.widgets.text("base_url", "https://dummyjson.com/users", "Base URL")
dbutils.widgets.text("limit", "10", "Limit (max rows returned)")
dbutils.widgets.text("skip", "0", "Skip (rows to offset)")
dbutils.widgets.dropdown("select", "", ["", "id", "firstName", "lastName", "email", "username", "phone", "gender", "city"], "Select fields (optional)")

base_url = dbutils.widgets.get("base_url")
limit = dbutils.widgets.get("limit")
skip = dbutils.widgets.get("skip")
select = dbutils.widgets.get("select")

# Build the query string from widget values
params = {"limit": limit, "skip": skip}
if select:
    params["select"] = select

response = requests.get(base_url, params=params, timeout=30)
response.raise_for_status()
data = response.json()

print(f"Request URL: {response.url}")
print(f"Total users available: {data.get('total', 'n/a')}")

# Flatten users into a pandas DataFrame for exploration
import pandas as pd

users = pd.json_normalize(data["users"])
display(users)

In [0]:
import requests
from datetime import date

# Date range for the dynamic filename
start_date = "2024-01-01"
end_date = "2024-12-31"

# API endpoint to fetch JSON data from
url = "https://dummyjson.com/users"

# Volume directory in Unity Catalog
volume_dir = "/Volumes/my/default/my"

response = requests.get(url, timeout=30)
response.raise_for_status()  # raises HTTPError for 4xx/5xx responses

# Build the target path using the date range
filename = f"users_{start_date}_{end_date}.json"
target_path = f"{volume_dir}/{filename}"

# Save the raw response payload to the Unity Catalog Volume
with open(target_path, "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"Saved raw JSON payload to: {target_path}")

In [0]:
from pyspark.sql import functions as F

# Path to the Unity Catalog Volume containing users_*.json files
volume_path = "/Volumes/my/default/my"

# Read all matching JSON files as raw, whole-text records
raw_df = (
    spark.read.text(
        f"{volume_path}/users_*.json",
        wholetext=True,
    )
    .withColumn("users", F.col("_metadata.file_name"))
    .withColumn("users_file_data", F.col("value"))
    .withColumn("users_ingestion", F.current_timestamp())
    .select("users", "users_file_data", "users_ingestion")
)

# Append the raw records into the Delta table
raw_df.write.format("delta").mode("append").saveAsTable("my.default.raw_users")

print(f"Successfully appended {raw_df.count()} file record(s) to my.default.raw_users")

display(spark.table("my.default.raw_users"))


In [0]:
%sql
select * from my.default.raw_users

In [0]:
from pyspark.sql import functions as F

# Read the raw users records (one row per ingested JSON file)
raw_df = spark.table("my.default.raw_users")

# Infer the JSON schema of the payload across all ingested files
schema_row = raw_df.select(F.expr("schema_of_json_agg(users_file_data)")).first()
payload_ddl = schema_row[0] if schema_row is not None else None
if payload_ddl is None:
    raise ValueError("No JSON payloads found in my.default.raw_users")
print(f"Inferred payload schema: {payload_ddl}")

# Parse the whole-text JSON payload, then explode the users array into one row per user
bronze_df = (
    raw_df
    .withColumn("payload", F.from_json("users_file_data", payload_ddl))
    .select(
        F.col("users").alias("source_file"),
        F.col("users_ingestion"),
        F.explode("payload.users").alias("user"),
    )
    .select("source_file", "users_ingestion", "user.*")
)

# Append the parsed user records to the bronze table
bronze_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("my.default.bronze_users")

print(f"Successfully appended {bronze_df.count()} user record(s) to my.default.bronze_users")

display(spark.table("my.default.bronze_users"))


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType

# Read the bronze users table
bronze_df = spark.table("my.default.bronze_users")

# Convert any struct/array (JSON-format) columns into JSON strings
json_cols = [
    field.name
    for field in bronze_df.schema.fields
    if isinstance(field.dataType, (StructType, ArrayType))
]
print(f"Columns converted to JSON strings: {json_cols if json_cols else 'none'}")

for col_name in json_cols:
    bronze_df = bronze_df.withColumn(col_name, F.to_json(F.col(col_name)))

# Overwrite the bronze table with the stringified columns
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("my.default.bronze_users")
)

print(f"Successfully rewrote my.default.bronze_users with {len(json_cols)} JSON column(s) converted to strings")

display(spark.table("my.default.bronze_users"))


In [0]:
%sql
select * from my.default.bronze_users

In [0]:
from pyspark.sql import functions as F

# Read the bronze users table
bronze_df = spark.table("my.default.bronze_users")

# Uppercase first and last names; replace empty maidenName values with NULL
silver_df = (
    bronze_df
    .withColumn("firstName", F.upper(F.col("firstName")))
    .withColumn("lastName", F.upper(F.col("lastName")))
    .withColumn(
        "maidenName",
        F.when(
            (F.col("maidenName").isNull()) | (F.trim(F.col("maidenName")) == ""),
            F.lit(None),
        ).otherwise(F.col("maidenName")),
    )
)

# Overwrite the silver table with the transformed records
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("my.default.silver_users")
)

print(f"Successfully wrote {silver_df.count()} record(s) to my.default.silver_users")

display(spark.table("my.default.silver_users"))


In [0]:
%sql
select count(*) from my.default.silver_users

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read the silver users table
silver_df = spark.table("my.default.silver_users")

# Deduplicate: keep one row per user id, preferring the most recently ingested record
window_spec = Window.partitionBy("id").orderBy(F.col("users_ingestion").desc())
deduped_df = (
    silver_df
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

row_count_before = silver_df.count()
row_count_after = deduped_df.count()
print(f"Silver rows: {row_count_before}, deduplicated rows: {row_count_after} ({row_count_before - row_count_after} duplicate(s) removed)")

# Create the gold table if it does not exist yet
spark.sql("""
    CREATE TABLE IF NOT EXISTS my.default.gold_users
    AS SELECT * FROM my.default.silver_users WHERE 1 = 0
""")

# Register the deduplicated records as the MERGE source view
deduped_df.createOrReplaceTempView("silver_users_deduped")

# Merge (upsert) deduplicated silver records into the gold table keyed on user id
merge_result = spark.sql("""
    MERGE INTO my.default.gold_users AS target
    USING silver_users_deduped AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

num_merged_rows = merge_result.count()
print(f"MERGE complete: {num_merged_rows} row(s) upserted into my.default.gold_users")

print("Duplicate ids remaining in gold (should be 0):")
display(
    spark.sql("""
        SELECT id, COUNT(*) AS cnt
        FROM my.default.gold_users
        GROUP BY id
        HAVING COUNT(*) > 1
    """)
)

display(spark.table("my.default.gold_users"))


# ASRITHA ETL NOTEBOOK — Medallion Architecture Summary

## Overview
This notebook implements a **batch ETL pipeline (Extract → Transform → Load)** following the **Medallion Architecture** (Raw → Bronze → Silver → Gold) for user data ingested from a public REST API.

## ETL / Medallion Flow

| Stage | Step | What Happens |
|---|---|---|
| **Extract** | API exploration | Query the users API with widgets (`limit`, `skip`, `select`) and preview results in pandas |
| **Extract** | API ingest | Fetch full JSON payload and save raw JSON file to a Unity Catalog Volume |
| **Raw** | Landing | Read `users_*.json` files as whole-text records with file metadata into the raw table |
| **Bronze** | Parse | Infer JSON schema, parse payload, explode the users array → one row per user |
| **Silver** | Clean | Normalize names, handle missing values |
| **Gold** | Curate | Deduplicate and merge into final conformed table |

## Key Locations

- **Source URL (API)**: `https://dummyjson.com/users` (dummyjson.com users API)
- **Volume source path (landing)**: `/Volumes/my/default/my` (raw JSON files named `users_{start_date}_{end_date}.json`)
- **Raw table**: `my.default.raw_users`
- **Bronze table**: `my.default.bronze_users`
- **Silver table**: `my.default.silver_users`
- **Gold table**: `my.default.gold_users`

## Transformations Applied

1. **Raw (landing)**: Read JSON files with `wholetext=True`, capture file name (`users`) and ingestion timestamp (`users_ingestion`); append one row per file to `raw_users`.
2. **Bronze (parse)**: Infer schema with `schema_of_json_agg(users_file_data)`; parse with `from_json` and `explode(payload.users)` → one row per user; convert struct/array columns to JSON strings; append to `bronze_users`.
3. **Silver (clean)**:
   - Uppercase `firstName` and `lastName`
   - Replace empty/NULL `maidenName` with NULL
   - Overwrite `silver_users` on each run

## Merge / Dedup Logic (Gold)

- Deduplicate silver records with `row_number()` over a window partitioned by `id`, ordered by `users_ingestion` DESC — keeping the **most recently ingested record per user id**.
- **MERGE (upsert)** into `my.default.gold_users`:
  ```sql
  MERGE INTO my.default.gold_users AS target
  USING silver_users_deduped AS source
  ON target.id = source.id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
  ```
- This guarantees `gold_users` always holds **one current row per user id** with no duplicates.
